# Chapter 9 · Machine Learning Interatomic Potentials — Colab notebook

Back to the chapter: <https://dongzhaohe321418-lab.github.io/materials-simulation-handbook/ch09-mlip/>

We train a tiny MACE potential end to end on a small synthetic dataset and check its parity against the reference. The dataset and model are deliberately small so the run finishes in a few minutes on a Colab GPU; the chapter's production recipe uses the same code with a larger dataset, more epochs and the official `mace_run_train` entry point.

This notebook is meant for **Google Colab** rather than the in-browser JupyterLite kernel, because it needs heavy packages (and, where noted, a GPU) that cannot run under Pyodide. Open it in Colab, and where the install cell mentions it, switch the runtime to a GPU via **Runtime -> Change runtime type -> GPU** before running the rest.


## Install

`mace-torch` pulls in the MACE library and its `e3nn` dependency. `torch` itself comes pre-installed on Colab, so it is not listed here. **Set the runtime to GPU** (Runtime -> Change runtime type -> GPU) before running the training cell — MACE training on CPU is painfully slow.

In [ ]:
!pip install mace-torch


## Check the GPU

MACE training needs a GPU for a reasonable turnaround. If this cell fails, switch the Colab runtime type to GPU and re-run from the top.

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU required: Runtime -> Change runtime type -> GPU'
print('device:', torch.cuda.get_device_name(0))


## Build a tiny training dataset

A real MACE study reads thousands of DFT-labelled structures from an extended-XYZ file. To keep this notebook self-contained and fast, we generate a small set of rattled bulk-copper cells and label them with ASE's cheap EMT calculator standing in for DFT. The workflow — write `train.xyz` / `valid.xyz` / `test.xyz` — is exactly the chapter's.

In [ ]:
import numpy as np
import ase.io
from ase.build import bulk
from ase.calculators.emt import EMT

rng = np.random.default_rng(0)
frames = []
base = bulk('Cu', crystalstructure='fcc', a=3.61, cubic=True).repeat((2, 2, 2))
for _ in range(120):
    atoms = base.copy()
    atoms.rattle(stdev=0.08, rng=rng)
    atoms.calc = EMT()
    atoms.info['energy'] = atoms.get_potential_energy()
    atoms.arrays['forces'] = atoms.get_forces()
    atoms.calc = None
    frames.append(atoms)

idx = rng.permutation(len(frames))
ase.io.write('train.xyz', [frames[i] for i in idx[:90]])
ase.io.write('valid.xyz', [frames[i] for i in idx[90:105]])
ase.io.write('test.xyz',  [frames[i] for i in idx[105:]])
print('wrote 90 train / 15 valid / 15 test structures')


## Train a small MACE model

We call MACE's official training entry point with a small architecture (two interaction layers, modest hidden dimension) and a short epoch budget. This is the same `mace_run_train` command the chapter uses for production, only with smaller knobs so it finishes quickly.

In [ ]:
import sys
from mace.cli.run_train import main as mace_run_train

args = [
    '--name', 'cu_mace',
    '--train_file', 'train.xyz',
    '--valid_file', 'valid.xyz',
    '--test_file', 'test.xyz',
    '--energy_key', 'energy',
    '--forces_key', 'forces',
    '--model', 'MACE',
    '--r_max', '5.0',
    '--num_interactions', '2',
    '--hidden_irreps', '32x0e + 32x1o',
    '--num_radial_basis', '8',
    '--max_ell', '2',
    '--correlation', '3',
    '--batch_size', '5',
    '--max_num_epochs', '30',
    '--energy_weight', '1.0',
    '--forces_weight', '100.0',
    '--lr', '0.01',
    '--device', 'cuda',
    '--default_dtype', 'float64',
    '--seed', '1',
]
sys.argv = ['mace_run_train'] + args
mace_run_train()


## Evaluate the trained potential

We load the saved model with the `MACECalculator` — the same object you would attach to run MD — and compare its per-atom energies against the reference labels on the held-out test set.

In [ ]:
from mace.calculators import MACECalculator

calc = MACECalculator(model_paths=['cu_mace.model'], device='cuda',
                      default_dtype='float64')

test_frames = ase.io.read('test.xyz', index=':')
e_pred, e_ref = [], []
for atoms in test_frames:
    ref = atoms.info['energy'] / len(atoms)
    atoms.calc = calc
    e_pred.append(atoms.get_potential_energy() / len(atoms))
    e_ref.append(ref)
e_pred, e_ref = np.array(e_pred), np.array(e_ref)
mae = np.mean(np.abs(e_pred - e_ref)) * 1000
print(f'energy MAE = {mae:.2f} meV/atom')


## Parity plot

The hallmark figure of every MLIP paper: predicted versus reference energy. Points hugging the diagonal mean the potential has learned the energy surface.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(e_ref, e_pred, s=20, alpha=0.7)
lim = [min(e_ref.min(), e_pred.min()), max(e_ref.max(), e_pred.max())]
ax.plot(lim, lim, 'k--', lw=0.8)
ax.set_xlabel('reference energy (eV/atom)')
ax.set_ylabel('MACE energy (eV/atom)')
ax.set_aspect('equal')
ax.set_title('MACE parity on held-out test set')
plt.show()
